# Projeto: Modelo Preditivo de Evolução de Casos de Dengue

**Autores:** Andressa, Celso e Helena
**Disciplina:** Data Science

## Objetivos
Este projeto utiliza Redes Neurais para prever se pacientes notificados com dengue serão hospitalizados, focando especialmente em comorbidades e sintomas clínicos. O objetivo é criar um modelo de triagem que possa ser utilizado no momento da entrada do paciente no sistema de saúde.

# Preparação dos Dados

## Importando bibliotecas

- **pandas** → manipulação e análise de dados em DataFrames.  
- **numpy** → cálculos numéricos com vetores e matrizes.  
- **seaborn** → visualizações estatísticas com boa estética.  
- **matplotlib.pyplot** → criação de gráficos personalizados.  
- **sklearn** → conjunto de ferramentas para Machine Learning, incluindo algoritmos de classificação, regressão, clustering, redução de dimensionalidade e pré-processamento de dados.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import chi2, f_classif

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Carregando os dados para um DataFrame

In [ ]:
path = "/content/drive/MyDrive/Projeto Data Science & Redes Neurais"

In [ ]:
# df = pd.read_csv("dados/DENGBR21_25.csv", sep=";", low_memory=False)
df = pd.read_csv("dados/DENGBR21_25_BALANCEADO.csv", low_memory=False)

In [ ]:
print(f"Total de linhas: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")

## Limpeza de Colunas Irrelevantes ou Incompletas
Para garantir que o modelo foque apenas em dados informativos, removemos:

* **Constantes:** Colunas que possuem um único valor (ex: um campo que é "Dengue" para todos os registros). Se não há variação, não há aprendizado.
* **Colunas Esparsas:** Variáveis com mais de 90% de ausência de dados. Manter colunas quase vazias forçaria o modelo a trabalhar com informações fragmentadas que não representam a realidade da amostra total.

A remoção de colunas administrativas antes do drop de linhas reduziu o dataset para 84k registros, porém garantiu que apenas casos com alta densidade de informações clínicas fossem mantidos, eliminando o ruído de notificações incompletas.

In [ ]:
def show_missing_stats(df):
  missing_by_row = df.isna().mean(axis=1) * 100
  print("---------- Missing por Linhas ----------")
  print(missing_by_row.describe())
  print("----------------------------------------\n")

  missing_by_col = (df.isna().sum() / df.shape[0]).sort_values(ascending=False)
  print("---------- Missing por Colunas ---------")
  print(missing_by_col)
  print("----------------------------------------\n")

In [ ]:
show_missing_stats(df)

In [ ]:
cols_to_remove = ["ID_REGIONA", "ID_UNIDADE", "ID_RG_RESI", "ID_PAIS"
                  , "TP_SISTEMA", "DT_DIGITA", "CS_FLXRET", "TPAUTOCTO",
                   "COUFINF", "UF", "COPAISINF", "SEM_NOT", "SEM_PRI", "ANO_NASC", "DT_ENCERRA", "EVOLUCAO", "DT_INTERNA","MUNICIPIO"]

for col in df.columns:
  if len(df[col].value_counts(dropna=False)) == 1 or df[col].isna().sum() / df.shape[0] > 0.60:
    cols_to_remove.append(col)

print("Colunas que serão removidas:", len(cols_to_remove))
print(cols_to_remove, "\n")

# rows_to_remove = df[df.isna().mean(axis=1) > 0.5].index
# print("Quantidade de linhas que serão removidas:", len(rows_to_remove))
# df = df.drop(columns=cols_to_remove, index=rows_to_remove)

df = df.drop(columns=cols_to_remove)

In [ ]:
show_missing_stats(df)

## Cálculo de Intervalos Temporais (Deltas)

* **`DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO`**: Mede o atraso entre o início da doença e a chegada ao sistema de saúde (importante para prever gravidade).

Removemos as variáveis de "Investigação", visto que em cerca de 90% dos registros, a data bate com a variável de "Notificação".

Também removemos variável "Classificação Final" para evitar o uso de dados que não estariam disponíveis no momento da triagem inicial.

Com isso, mantivemos apenas a coluna **`DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO`**.


In [ ]:
df["DT_NOTIFIC"] = pd.to_datetime(df["DT_NOTIFIC"])
df["DT_SIN_PRI"] = pd.to_datetime(df["DT_SIN_PRI"])
df["DT_INVEST"] = pd.to_datetime(df["DT_INVEST"])

df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"] = (df["DT_NOTIFIC"] - df["DT_SIN_PRI"]).dt.days
print("------------------------------------------------------------")
print(df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"].value_counts().head())
print("------------------------------------------------------------\n")

df["DELTA_NOTIFICACAO_INVESTIGACAO"] = df["DT_INVEST"] - df["DT_NOTIFIC"]
print("------------------------------------------------------------")
print(df["DELTA_NOTIFICACAO_INVESTIGACAO"].value_counts().head())
print("------------------------------------------------------------\n")

df = df.drop(columns=["DELTA_NOTIFICACAO_INVESTIGACAO", "DT_INVEST", "DT_NOTIFIC", "DT_SIN_PRI", "CLASSI_FIN"])

## Limpeza Geográfica

Nesta etapa, validamos a necessidade de manter múltiplas colunas de localização (Residência, Notificação e Infecção).

* **Residência vs. Notificação**: Identificamos que em cerca de 94% dos casos de Município e 99% de UF, o local de residência é o mesmo da notificação. Optamos por manter apenas os dados de **Notificação** por estarem 100% preenchidos e representarem o local do atendimento médico.
* **Município de Infecção (`COMUNINF`)**: descartado dado que o perfil de infraestrutura de saúde e a capacidade de absorção de leitos clínicos estão diretamente vinculados ao local de atendimento/notificação do paciente, e não ao local de infecção.

In [ ]:
df["SG_UF_NOT"] = df["SG_UF_NOT"].str.replace("sp", "35")

print("---------- Município de Notificação == Residência  ---------")
print((df["ID_MUNICIP"] == df["ID_MN_RESI"]).value_counts())
print("------------------------------------------------------------\n")

print("------------- UF de Notificação == Residência  -------------")
print((df["SG_UF_NOT"].astype(str) == df["SG_UF"].astype(str)).value_counts())
print("------------------------------------------------------------\n")

print("----------- Município de Notificação == Infecção  ----------")
print((df["ID_MUNICIP"] == df["COMUNINF"]).value_counts())
print("------------------------------------------------------------\n")

print("------------ Município de Infecção mais comuns  ------------")
print(df["COMUNINF"].value_counts().head())
print("------------------------------------------------------------\n")

df = df.drop(columns=["ID_MN_RESI", "SG_UF", "COMUNINF"])

## Normalização da Variável de Idade (`NU_IDADE_N`)

O campo de idade no sistema SINAN utiliza uma codificação composta de 4 dígitos, onde o primeiro dígito representa a unidade de medida e os três últimos representam o valor:
* **1**: Horas | **2**: Dias | **3**: Meses | **4**: Anos

**Importância da Transformação:**

Para um modelo, os valores brutos (ex: `3009` para 9 meses e `4020` para 20 anos) criariam uma distorção matemática, pois o modelo interpretaria o código da unidade como parte da magnitude. Ao converter tudo para **Anos Decimais (float)**, estabelecemos uma escala linear biológica correta, permitindo que o modelo aprenda o risco real associado a cada faixa etária (especialmente o alto risco em neonatos e idosos).

**Tratamento de Inconsistências:**
Utilizamos o método `.zfill(4)` para garantir que registros preenchidos fora do padrão recebam zeros à esquerda e sejam processados corretamente pela lógica de prefixos.

In [ ]:
def transform_ages(age_code):
  age_code_str = str(int(age_code)).zfill(4)

  prefix = age_code_str[0]
  value = float(age_code_str[1:])

  if prefix == "1":
    return value / (24 * 365)
  elif prefix == "2":
    return value / 365
  elif prefix == "3":
    return value / 12
  elif prefix == "4":
    return value
  else:
    return value

df["NU_IDADE_N_ANOS"] = df["NU_IDADE_N"].apply(transform_ages)
df["NU_IDADE_N_ANOS"].describe()
df = df.drop(columns=["NU_IDADE_N"])

## Removendo Exames Laboratoriais Irregulares

criteria_mapping = {
    0: "CLINICO",
    1: "LABORATORIO"
}

In [ ]:
exams = ["RESUL_SORO", "RESUL_NS1", "RESUL_VI_N", "RESUL_PCR_", "HISTOPA_N", "IMUNOH_N"]

# Quantificar os dados no DataFrame com base nos critérios - laboraorial e clínico - e/ou lista de exames
# Casos clínicos: CRITERIO = 2 ou 3
# Casos laboratoriais: CRITERIO = 1
# Exames laboratoriais positivos: qualquer exame da lista de exames com resultado positivo (1)
# Casos laboratoriais com exames realizados: CRITERIO = 1 e pelo menos um exame da lista de exames positivo
# Casos laboratoriais sem exames realizados: CRITERIO = 1 e nenhum exame da lista de exames positivo
# A partir dessa análise, é possível identificar a quantidade de casos clínicos e laboratoriais, bem como a proporção de casos laboratoriais que realizaram exames e aqueles que não realizaram exames, o que pode ser útil para entender melhor o perfil dos pacientes e a qualidade dos dados disponíveis.

clinic_cases = ((df["CRITERIO"] == 2) | (df["CRITERIO"] == 3))
n_laboratorial_cases = (df["CRITERIO"] == 1).sum()
ignored_cases = ((df["CRITERIO"] == 0) | (df["CRITERIO"] == 9) | (df["CRITERIO"].isna()))
laboratorial_positive_exams = (df[exams] == 1).any(axis=1)
laboratorial_cases_with_exams = ((df["CRITERIO"] == 1) & (laboratorial_positive_exams))
laboratorial_cases_without_exams = ((df["CRITERIO"] == 1) & (~laboratorial_positive_exams))

print(" ------------------ AVALIAÇÃO ESTATÍSTICA --------------------")
print(f"\nQuantidade de casos laboratoriais: {n_laboratorial_cases}.")
print(f"Quantidade de casos laboratoriais com exame realizado: {laboratorial_cases_with_exams.sum()}.")
print(f"Quantidade de casos laboratoriais sem exame realizado: {laboratorial_cases_without_exams.sum()}.")
print(f"Quantidade de casos clínicos: {clinic_cases.sum()}.")
print(f"Quantidade de casos rotulados como 'Ignorado': {ignored_cases.sum()}.\n")

# Percebe-se que 8478 casos laboratoriais não apresentaram exame realizado. Dessa forma, iremos remover esses casos irregulares. Além disso, também agrupamos os casos clínicos, transformamos os casos onde o critério era Ignorado (0 ou 9) em NaN e trocamos o identificados dos casos clínicos para 0.
valid_cases = clinic_cases | laboratorial_cases_with_exams | ignored_cases
df = df[valid_cases]
df["CRITERIO"] = df["CRITERIO"].replace([0, 9], np.nan).replace([2, 3], 0)
print(df["CRITERIO"].value_counts(dropna=False))

# Após verificar que os casos clínicos e laboratoriais foram regularizados, podemos remover as colunas dos resultados de exames, pois todos os pacientes que sobraram foram diagnosticados com dengue.
df = df.drop(columns=exams)

## Transformando os Dados Geográficos

In [ ]:
df_adh = pd.read_csv("dados/ONU_ADH_MUNICIPIOS.csv")
df_convert = pd.read_csv("dados/CONVERSAO_IDS_MUNICIPIOS.csv")

df_adh = df_adh.merge(df_convert, on="id_municipio", how="left")
cols_merge = ["id_municipio_6", "idhm_l", "expectativa_vida", "prop_pobreza"]
df_adh = df_adh.loc[df_adh["ano"] == 2010, cols_merge]

df = df.merge(df_adh, left_on="ID_MUNICIP", right_on="id_municipio_6", how="left")
df = df.drop(columns=["SG_UF_NOT", "ID_MUNICIP", "id_municipio_6"])

## Tratamento e Limpeza das Variáveis Categóricas

In [ ]:
df["CS_GESTANT"] = df["CS_GESTANT"].replace([2, 3, 4], 1).replace([5, 6], 0).replace(9, np.nan)

df["CS_SEXO"] = df["CS_SEXO"].replace("I", np.nan).replace("M", 0).replace("F", 1)

race_mapping = {
    1: "BRANCA",
    2: "PRETA",
    3: "AMARELA",
    4: "PARDA",
    5: "INDIGENA",
    9: "IGNORADO",
}
df["CS_RACA"] = df["CS_RACA"].replace(race_mapping)

scholarity_mapping = {
    0: "ANALFABETO",
    1: "FUNDAMENTAL_INCOMPLETO",
    2: "FUNDAMENTAL_INCOMPLETO",
    3: "FUNDAMENTAL_INCOMPLETO",
    4: "FUNDAMENTAL_COMPLETO",
    5: "MEDIO_INCOMPLETO",
    6: "MEDIO_COMPLETO",
    7: "SUPERIOR_INCOMPLETO",
    8: "SUPERIOR_COMPLETO",
    9: "IGNORADO",
    10: "MENOR_7_ANOS",
}
df["CS_ESCOL_N"] = df["CS_ESCOL_N"].replace(scholarity_mapping)

cols_clinic_signals = [
    "FEBRE", "MIALGIA", "CEFALEIA", "EXANTEMA", "VOMITO", "NAUSEA", "DOR_COSTAS", "CONJUNTVIT", "ARTRITE", "ARTRALGIA", "PETEQUIA_N", 
    "LEUCOPENIA", "LACO", "DOR_RETRO", "DIABETES", "HEMATOLOG", "HEPATOPAT", "RENAL", "HIPERTENSA", "ACIDO_PEPT", "AUTO_IMUNE"
]
for col in cols_clinic_signals:
    df[col] = df[col].replace(2, 0).replace(9, np.nan)
    
df["HOSPITALIZ"] = df["HOSPITALIZ"].replace(2, 0).replace(9, np.nan)

## Renomeando Atributos

In [ ]:
cols = {
    "NU_ANO": "ANO",
    "CS_SEXO": "SEXO", 
    "CS_GESTANT": "GESTANTE", 
    "CS_RACA": "RACA", 
    "CS_ESCOL_N": "ESCOLARIDADE", 
    "CONJUNTVIT": "CONJUNTIVITE", 
    "PETEQUIA_N": "PETEQUIAS", 
    "DOR_RETRO": "DOR_RETROORBITAL", 
    "HEMATOLOG": "DOENCAS_HEMATOLOGICAS", 
    "HEPATOPAT": "HEPATOPATIAS", 
    "RENAL": "DOENCA_RENAL",
    "HIPERTENSA": "HIPERTENSAO", 
    "ACIDO_PEPT": "DOENCA_ACIDO_PEPTICA", 
    "AUTO_IMUNE": "DOENCAS_AUTO_IMUNE", 
    "HOSPITALIZ": "HOSPITALIZACAO", 
    "CRITERIO": "CRITERIO_CONFIRMACAO",
    "NU_IDADE_N_ANOS": "IDADE", 
    "idhm_l": "IDH_MUNICIPAL_LONGEVIDADE",
    "expectativa_vida": "EXPECTATIVA_VIDA", 
    "prop_pobreza": "PROPORCAO_POBRES"
}
df = df.rename(columns=cols)

In [ ]:
print(len(df.columns))
print(df.columns)

show_missing_stats(df)

In [ ]:
df.to_csv("dados/BASE.csv", index=False)

# Pré-Processamento dos Dados

## Descrição da Base

A base de dados utilizada neste trabalho é composta por 33 atributos que reúnem informações clínicas, demográficas e socioeconômicas dos pacientes. Para garantir uma correta modelagem e estruturação do pipeline de mineração de dados, as variáveis foram mapeadas e classificadas de acordo com suas propriedades matemáticas, dividindo-se entre os tipos numérico e categórico, além de suas respectivas escalas de mensuração (nominal ou razão) e níveis de cardinalidade (binária, discreta ou contínua), conforme apresentado na tabela a seguir:

| Coluna | Tipo | Escala | Cardinalidade |
| :--- | :---: | :---: | :---: |
| ANO | Numérica | Razão | Discreta |
| SEXO | Categórica | Nominal | Binária |
| GESTANTE | Categórica | Nominal | Binária |
| RACA | Categórica | Nominal | Discreta |
| ESCOLARIDADE | Categórica | Nominal | Discreta |
| FEBRE | Categórica | Nominal | Binária |
| MIALGIA | Categórica | Nominal | Binária |
| CEFALEIA | Categórica | Nominal | Binária |
| EXANTEMA | Categórica | Nominal | Binária |
| VOMITO | Categórica | Nominal | Binária |
| NAUSEA | Categórica | Nominal | Binária |
| DOR_COSTAS | Categórica | Nominal | Binária |
| CONJUNTIVITE | Categórica | Nominal | Binária |
| ARTRITE | Categórica | Nominal | Binária |
| ARTRALGIA | Categórica | Nominal | Binária |
| PETEQUIAS | Categórica | Nominal | Binária |
| LEUCOPENIA | Categórica | Nominal | Binária |
| LACO | Categórica | Nominal | Binária |
| DOR_RETROORBITAL | Categórica | Nominal | Binária |
| DIABETES | Categórica | Nominal | Binária |
| DOENCAS_HEMATOLOGICAS | Categórica | Nominal | Binária |
| HEPATOPATIAS | Categórica | Nominal | Binária |
| DOENCA_RENAL | Categórica | Nominal | Binária |
| HIPERTENSAO | Categórica | Nominal | Binária |
| DOENCA_ACIDO_PEPTICA | Categórica | Nominal | Binária |
| DOENCAS_AUTO_IMUNE | Categórica | Nominal | Binária |
| HOSPITALIZACAO | Categórica | Nominal | Binária |
| CRITERIO_CONFIRMACAO | Categórica | Nominal | Binária |
| DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO | Numérica | Razão | Discreta |
| IDADE | Numérica | Razão | Discreta |
| IDH_MUNICIPAL_LONGEVIDADE | Numérica | Razão | Contínua |
| EXPECTATIVA_VIDA | Numérica | Razão | Contínua |
| PROPORCAO_POBRES | Numérica | Razão | Contínua |

In [ ]:
df = pd.read_csv("dados/BASE.csv")
df = df.dropna(subset=["HOSPITALIZACAO"]).copy()
df.info()

## Análise Estatística dos Atributos

### Atributos Numéricos

| Estatística | DELTA SINTOMAS-NOTIFICAÇÃO | IDADE | IDH MUNICIPAL LONGEVIDADE | EXPECTATIVA DE VIDA | PROPORÇÃO DE POBRES |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Contagem (*count*)** | 315.808 | 315.808 | 315.770 | 315.770 | 315.770 |
| **Média (*mean*)** | 4,72 dias | 35,05 | 0,841 | 75,46 anos | 8,00% |
| **Desvio Padrão (*std*)** | 10,98 dias | 19,76 | 0,029 | 1,73 anos | 9,34% |
| **Mínimo (*min*)** | 0 | 0 | 0,672 | 65,30 anos | 0,00% |
| **25% (1º Quartil)** | 1,00 dia | 20,00 | 0,829 | 74,74 anos | 3,16% |
| **50% (Mediana)** | 3,00 dias | 33,00 | 0,844 | 75,66 anos | 4,59% |
| **75% (3º Quartil)** | 5,00 dias | 49,00 | 0,858 | 76,46 anos | 8,10% |
| **Máximo (*max*)** | 374 dias | 406 | 0,894 | 78,64 anos | 78,59% |

- **Outliers Identificados:** Observa-se que, mesmo após a filtragem da variável alvo, o valor máximo para o atributo `IDADE` consta como **406** e para o delta cronológico consta como **374 dias**. Ambas as métricas representam inconsistências claras de preenchimento ou digitação no sistema original que serão mitigadas pelas quebras da árvore de decisão.

### Atributos Categóricos e Binários (Frequências)

* **ANO DO CASO** (Total: 315.808)
  * 2023: 72.405 registros (~22,9%)
  * 2022: 71.802 registros (~22,7%)
  * 2024: 69.377 registros (~22,0%)
  * 2025: 52.860 registros (~16,7%)
  * 2021: 49.123 registros (~15,6%)
  * 2026: 241 registros (~0,1%)

* **SEXO** (Total válido: 315.515 | Nulos: 293)
  * **1.0 (Feminino / Mapeado):** 170.447 (~54,0%)
  * **0.0 (Masculino / Mapeado):** 145.068 (~46,0%)

* **GESTANTE** (Total válido: 292.675 | Nulos: 23.133)
  * **0.0 (Não):** 290.037 (~99,1%)
  * **1.0 (Sim):** 2.638 (~0,9%)

* **RAÇA / COR** (Total válido: 315.806 | Nulos: 2)
  * BRANCA: 139.559 (~44,2%)
  * PARDA: 111.214 (~35,2%)
  * IGNORADO: 48.018 (~15,2%)
  * PRETA: 12.790 (~4,1%)
  * AMARELA: 3.573 (~1,1%)
  * INDÍGENA: 652 (~0,2%)

* **ESCOLARIDADE** (Total válido: 267.247 | Nulos: 48.561)
  * IGNORADO: 99.968 (~37,4%)
  * MÉDIO COMPLETO: 57.009 (~21,3%)
  * FUNDAMENTAL INCOMPLETO: 36.064 (~13,5%)
  * MENOR 7 ANOS: 21.694 (~8,1%)
  * MÉDIO INCOMPLETO: 17.189 (~6,4%)
  * SUPERIOR COMPLETO: 15.603 (~5,8%)
  * FUNDAMENTAL COMPLETO: 13.236 (~5,0%)
  * SUPERIOR INCOMPLETO: 5.015 (~1,9%)
  * ANALFABETO: 1.469 (~0,6%)

### Manifestações Clínicas (Sintomas)

* **FEBRE**
  * **1.0 (Sim):** 263.485 (~85,6%) | **0.0 (Não):** 44.244 (~14,4%)
* **MIALGIA**
  * **1.0 (Sim):** 245.208 (~79,7%) | **0.0 (Não):** 62.521 (~20,3%)
* **CEFALEIA**
  * **1.0 (Sim):** 244.474 (~79,4%) | **0.0 (Não):** 63.255 (~20,6%)
* **EXANTEMA**
  * **0.0 (Não):** 269.140 (~87,5%) | **1.0 (Sim):** 38.589 (~12,5%)
* **VOMITO**
  * **0.0 (Não):** 230.693 (~75,0%) | **1.0 (Sim):** 77.036 (~25,0%)
* **NAUSEA**
  * **0.0 (Não):** 184.421 (~59,9%) | **1.0 (Sim):** 123.308 (~40,1%)
* **DOR_COSTAS**
  * **0.0 (Não):** 215.945 (~70,2%) | **1.0 (Sim):** 91.784 (~29,8%)
* **CONJUNTIVITE**
  * **0.0 (Não):** 296.271 (~96,3%) | **1.0 (Sim):** 11.458 (~3,7%)
* **ARTRITE**
  * **0.0 (Não):** 276.683 (~89,9%) | **1.0 (Sim):** 31.046 (~10,1%)
* **ARTRALGIA**
  * **0.0 (Não):** 249.129 (~81,0%) | **1.0 (Sim):** 58.600 (~19,0%)
* **PETEQUIAS**
  * **0.0 (Não):** 284.965 (~92,6%) | **1.0 (Sim):** 22.764 (~7,4%)
* **LEUCOPENIA**
  * **0.0 (Não):** 293.275 (~95,3%) | **1.0 (Sim):** 14.454 (~4,7%)
* **LACO (Prova do Laço)**
  * **0.0 (Não):** 297.165 (~96,6%) | **1.0 (Sim):** 10.564 (~3,4%)
* **DOR_RETROORBITAL**
  * **0.0 (Não):** 209.905 (~68,2%) | **1.0 (Sim):** 97.824 (~31,8%)

### Comorbidades e Histórico Clínico

* **DIABETES**
  * **0.0 (Não):** 295.704 (~96,1%) | **1.0 (Sim):** 12.019 (~3,9%)
* **DOENCAS_HEMATOLOGICAS**
  * **0.0 (Não):** 306.214 (~99,5%) | **1.0 (Sim):** 1.510 (~0,5%)
* **HEPATOPATIAS**
  * **0.0 (Não):** 306.177 (~99,5%) | **1.0 (Sim):** 1.547 (~0,5%)
* **DOENCA_RENAL**
  * **0.0 (Não):** 306.311 (~99,5%) | **1.0 (Sim):** 1.411 (~0,5%)
* **HIPERTENSAO**
  * **0.0 (Não):** 279.214 (~90,7%) | **1.0 (Sim):** 28.510 (~9,3%)
* **DOENCA_ACIDO_PEPTICA**
  * **0.0 (Não):** 306.166 (~99,5%) | **1.0 (Sim):** 1.556 (~0,5%)
* **DOENCAS_AUTO_IMUNE**
  * **0.0 (Não):** 305.833 (~99,4%) | **1.0 (Sim):** 1.889 (~0,6%)

### Desfecho e Confirmação

* **HOSPITALIZACAO (Variável Alvo)** (Total: 315.808)
  * **0.0 (Não Internado):** 302.210 (~95,7%)
  * **1.0 (Internado):** 13.598 (~4,3%)

* **CRITERIO_CONFIRMACAO** (Total válido: 305.223 | Nulos: 10.585)
  * **0.0 (Clínico-Epidemiológico / Mapeado):** 198.221 (~64,9%)
  * **1.0 (Laboratorial / Mapeado):** 107.002 (~35,1%)

In [ ]:
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
print(df[numeric_cols].describe(), "\n")

for col in df.columns:
    if col not in numeric_cols:
        print(df[col].value_counts(), "\n")

## Avaliação dos Resultados dos Processos de Data Mining no Modelo Decision Tree (DT)

In [ ]:
def unbalanced_train_test_split(df, test_size, prop_hosp, prop_non_hosp, random_state):
    df_hospitalized = df[df["HOSPITALIZACAO"] == 1]
    df_non_hospitalized = df[df["HOSPITALIZACAO"] == 0]
    
    num_test = int(len(df) * test_size)
    num_hospitalized_test = int(num_test * prop_hosp)
    num_non_hospitalized_test = int(num_test * prop_non_hosp)
    
    # Sorteio teste
    X_y_hospitalized_test = df_hospitalized.sample(n=num_hospitalized_test, random_state=random_state)
    X_y_non_hospitalized_test = df_non_hospitalized.sample(n=num_non_hospitalized_test, random_state=random_state)
    X_y_test = pd.concat([X_y_hospitalized_test, X_y_non_hospitalized_test])
    
    # Treino fica com o resto
    X_y_hospitalized_train = df_hospitalized.drop(X_y_hospitalized_test.index)
    X_y_non_hospitalized_train = df_non_hospitalized.drop(X_y_non_hospitalized_test.index)
    X_y_train = pd.concat([X_y_hospitalized_train, X_y_non_hospitalized_train])
    
    X_cols = [col for col in df.columns if col not in ["HOSPITALIZACAO"]]
    X_train, y_train = X_y_train[X_cols], X_y_train["HOSPITALIZACAO"]
    X_test, y_test = X_y_test[X_cols], X_y_test["HOSPITALIZACAO"]
    
    return X_train, X_test, y_train, y_test

In [ ]:
def run_dt_pipeline(df, experiment_name, normalize=False, encode=True):
    # Passar a lista de colunas categóricas para transformar
    numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
    categorical_cols = [col for col in df.columns if col not in numeric_cols + ["ANO", "HOSPITALIZACAO"]]
    if encode:
        df = pd.get_dummies(
            df, 
            columns=categorical_cols, 
            drop_first=True
        )

    # 1. Separar em treino e teste
    # X = df.drop(columns=["HOSPITALIZACAO", "ANO"])
    # y = df["HOSPITALIZACAO"]
    
    # X_train, X_test, y_train, y_test = train_test_split(
    #     X, y, test_size=0.2, random_state=42, stratify=y
    # )
    X_train, X_test, y_train, y_test = unbalanced_train_test_split(
        df.drop(columns=["ANO"]), test_size=0.2, prop_hosp=0.05, prop_non_hosp=0.95, random_state=42
    )
    
    if normalize:
        scaler = MinMaxScaler()
        
        X_train, X_test = X_train.copy(), X_test.copy()
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # 2. Definir o modelo base
    # dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
    dt = DecisionTreeClassifier(random_state=42)

    # 3. Definir a grade de hiperparâmetros
    # Os hiperparâmetros mais importantes para controlar complexidade e generalização do modelo costumam ser o max_depth, min_samples_split e min_samples_leaf.
    # O criterion define a medida de qualidade da divisão, com opções como gini e entropy nas versões atuais do scikit-learn
    param_grid = {
        "criterion": ["gini", "entropy"],        # Define a métrica de pureza / cálculo do caos dos nós
        "max_depth": [3, 5, 8],                  # Controla o tamanho da árvore / evita overfitting (IMPORTANTE)
        "min_samples_split": [10, 50],           # Evita divisões em grupos muito pequenos (IMPORTANTE)
        "min_samples_leaf": [5, 20],             # Garante tamanho mínimo de amostras por folha (IMPORTANTE)
        "splitter": ["best"],                    # Avalia a divisão matematicamente ideal
        "class_weight": [{0: 1, 1: 1}, {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}] 
    }

    # 4. Grid Search com Cross Validation
    # O F1-Score é superior porque, ao ignorar o sucesso fácil dos verdadeiros negativos, ele é diretamente impactado pelas quedas de Precisão e Recall, forçando o Grid Search a escolher um modelo que minimize os Falsos Negativos sem gerar um número absurdamente caótico de alarmes falsos.
    grid_search = GridSearchCV(
        estimator=dt, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1, verbose=1 
    )

    # 5. Ajustar o modelo aos dados de treino
    grid_search.fit(X_train, y_train)

    # 6. Melhor modelo encontrado
    best_model = grid_search.best_estimator_

    # 7. Previsões no conjunto de teste
    y_pred = best_model.predict(X_test)

    # 8. Resultados
    print("Melhores hiperparâmetros:")
    print(grid_search.best_params_)

    # cv_results_ guarda o desempenho de todas as combinações avaliadas pelo GridSearchCV
    # results = pd.DataFrame(grid_search.cv_results_)
    # cols = [
    #     "mean_test_score",
    #     "std_test_score",
    #     "param_criterion",
    #     "param_max_depth",
    #     "param_min_samples_split",
    #     "param_min_samples_leaf",
    #     # "param_splitter",
    # ]
    # print(results[cols].sort_values(by="mean_test_score", ascending=False).head(10))

    print("\nMelhor média do F1-Score na validação cruzada:")
    print(grid_search.best_score_)

    print("\nAcurácia no teste:")
    print(accuracy_score(y_test, y_pred))

    print("\nMatriz de confusão Teste:")
    conf_matrix = confusion_matrix(y_test, y_pred)
    conf_matrix_df = pd.DataFrame(
        conf_matrix, 
        index=["Real: Não Hospitalizado", "Real: Hospitalizado"], 
        columns=["Previsto: Não Hospitalizado", "Previsto: Hospitalizado"]
    ).astype(object)
    conf_matrix_df.iloc[0, 0] = f"{conf_matrix[0, 0]} (VN)"
    conf_matrix_df.iloc[0, 1] = f"{conf_matrix[0, 1]} (FP)"
    conf_matrix_df.iloc[1, 0] = f"{conf_matrix[1, 0]} (FN)"
    conf_matrix_df.iloc[1, 1] = f"{conf_matrix[1, 1]} (VP)"
    print(conf_matrix_df)

    print("\nRelatório de classificação Teste:")
    print(classification_report(y_test, y_pred, target_names=["Não Hospitalizado", "Hospitalizado"]))

    # 9. Visualização gráfica da melhor árvore
    plt.figure(figsize=(25, 10))
    plot_tree(
        best_model,
        feature_names=X_train.columns,
        class_names=["Não Hospitalizado", "Hospitalizado"],
        filled=True,
        rounded=True,
        fontsize=12,
    )
    plt.title(f"Melhor Árvore de Decisão Encontrada pelo GridSearchCV - {experiment_name}")
    plt.tight_layout()
    plt.show()

    # 11. Visualização textual das regras da árvore
    # rules = export_text(best_model, feature_names=list(X.columns))
    # print("\nRegras da melhor árvore:\n")
    # print(rules)
    
    return best_model

### Modelo na Base 0

In [ ]:
run_dt_pipeline(df, "Base 0")

### Modelo na Base 1 (Tratamento de Outliers e Missing Values)

A opção por realizar o tratamento de outliers numéricos e a imputação / filtragem de missing values de forma unificada na Base 1 justifica-se pelo fato de que ambas as inconsistências estão interligadas no contexto de registros hospitalares reais. Tratar tais fenômenos de forma isolada poderia mascarar vieses ou induzir a árvore de decisão ao overfitting baseado em dados corrompidos. A abordagem conjunta garantiu a integridade estatística do dataset, resultando em um ganho expressivo de Recall (de 47% para 62%) na classe de interesse (Hospitalizados).

#### Tratamento de Outliers

In [ ]:
# Outliers de colunas numéricas
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color="skyblue")
    axes[i].set_title(f"Distribuição de {col} (Detectando Outliers)")
    axes[i].set_ylabel("")
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

# Variáveis Municipais (IDH, Expectativa de Vida, Proporção de Pobres) nao mexer pq nao representam erro, mas sim a desigualdade real entre os municipios dos estados
# Tratar outliers DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO e IDADE que sao erros
mask_numerical = (df["IDADE"] >= 0) & (df["IDADE"] <= 120) \
                & (df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"] >= 0) & (df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"] <= 30)

# Outliers de colunas categóricas
categorical_cols = [col for col in df.columns if col not in numeric_cols + ["HOSPITALIZACAO", "ANO"]]

fig, axes = plt.subplots(9, 3, figsize=(22, 44))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, ax=axes[i], stat="percent", order=df[col].value_counts().index, palette="pastel", hue=col)
    axes[i].set_title(f"Distribuição por {col} (Detectando Outliers)")
    axes[i].tick_params(axis="x", rotation=45)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()
# n precisa remover nada, nada eh outlier apenas falta de representatividade

lines_before = len(df)
df_1 = df[mask_numerical]
lines_after = len(df_1)
lines_removed = lines_after - lines_before
print(" ------------------ RELATÓRIO DA LIMPEZA DE OUTLIERS --------------------")
print(f"Linhas antes do filtro: {lines_before}")
print(f"Linhas após o filtro:  {lines_after}")
print(f"Total de outliers removidos: {lines_removed} ({(lines_removed / lines_before) * 100 :.3f}%)")

#### Tratamento de Missing Values

In [ ]:
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
comorbidities_cols = [
    "GESTANTE", "FEBRE", "MIALGIA", "CEFALEIA", "EXANTEMA", "VOMITO", "NAUSEA", "DOR_COSTAS",
    "CONJUNTIVITE", "ARTRITE", "ARTRALGIA", "PETEQUIAS", "LEUCOPENIA", "LACO", 
    "DOR_RETROORBITAL", "DIABETES", "DOENCAS_HEMATOLOGICAS", "HEPATOPATIAS", 
    "DOENCA_RENAL", "HIPERTENSAO", "DOENCA_ACIDO_PEPTICA", "DOENCAS_AUTO_IMUNE"
]
categorical_cols = ["SEXO", "RACA", "ESCOLARIDADE", "CRITERIO_CONFIRMACAO"]

missing_per_row = df_1.isna().sum(axis=1)

plt.figure(figsize=(10, 6))
sns.histplot(missing_per_row, stat="percent", binwidth=1, color="teal", edgecolor="black", kde=False)
plt.title("Distribuição de Dados Ausentes por Registro (Paciente)", fontsize=14)
plt.xlabel("Quantidade de Atributos Missing na Linha", fontsize=12)
plt.ylabel("Quantidade de Registros (Linhas)", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

lines_before = len(df_1)
df_1 = df_1[missing_per_row < 15] # Mantendo apenas registros que possuem MENOS de 15 campos nulos

df_1[comorbidities_cols] = df_1[comorbidities_cols].fillna(0) # assumir que se n tem eh 0
df_1[categorical_cols] = df_1[categorical_cols].fillna("IGNORADO") # Nas categóricas, o que sobrou vira "IGNORADO"

# Nas numéricas, usamos a mediana
for col in  numeric_cols: 
    if df_1[col].isna().sum() > 0:
        df_1[col] = df_1[col].fillna(df_1[col].median())
lines_after = len(df_1)
lines_removed = lines_after - lines_before
print(" ------------------ RELATÓRIO DA LIMPEZA DE MISSING VALUES --------------------")
print(f"Linhas antes do filtro: {lines_before}")
print(f"Linhas após o filtro:  {lines_after}")
print(f"Total de linhas removidas: {lines_removed} ({(lines_removed / lines_before) * 100 :.3f}%)")

#### Avaliando o Resultado

In [ ]:
run_dt_pipeline(df_1, "Base 1 - Limpeza de Dados (Outliers e Missing Values)")

# A escolha da Base 1 (dados tratados) em detrimento da Base 0 (dados brutos/sujos) justifica-se pelo princípio da generalização e robustez metodológica. Embora a Base 0 apresente métricas de validação ligeiramente superiores em termos absolutos, tal fenômeno é decorrente do overfitting ao ruído, onde o algoritmo de árvore de decisão passa a codificar outliers e padrões arbitrários de dados ausentes (missing values) como regras preditivas validáveis

# Ao realizar a filtragem de registros severamente corrompidos e padronizar a imputação de sintomas e comorbidades de acordo com a semântica epidemiológica (assumindo a ausência do sintoma na falta de registro positivo), a Base 1 elimina esses sinais espúrios. O modelo resultante, portanto, reflete relações clínicas reais e sustentáveis, garantindo que o classificador mantenha sua capacidade preditiva estável e explicável quando submetido a cenários de produção no mundo real, livre dos vícios estatísticos presentes na base original.

df_1.to_csv("dados/BASE_v2.csv", index=False)

### Modelo na Base 2 (Normalização / Transformação)

In [ ]:
df_1 = pd.read_csv("dados/BASE_v2.csv")
df_2 = df_1.copy()
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df_1[col], kde=True, ax=axes[i], color="teal")
    axes[i].set_title(f"Distribuição de {col}")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Frequência")
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

# n segue gaussiana, entao usaremos MinMax! implementacao foi feita na funcao run_dt_pipeline()

In [ ]:
run_dt_pipeline(df_2, "Base 2 - Normalização / Transformação", normalize=True)
# nao muda nada para a base 1

### Modelo na Base 3 (Discretização)

In [ ]:
df_1 = pd.read_csv("dados/BASE_v2.csv")
df_3 = df_1.copy()

age_bins = [-1, 12, 60, df_1["IDADE"].max()]
age_labels = ["CRIANCA_ADOLESCENTE", "ADULTO", "IDOSO"]
df_3["IDADE_DISCRETA"] = pd.cut(df_3["IDADE"], bins=age_bins, labels=age_labels)

delta_bins = [-1, 6, 12, 18, 30]
delta_labels = ["NOTIFICACAO_RAPIDA", "NOTIFICACAO_MEDIA", "NOTIFICACAO_TARDIA", "NOTIFICACAO_EXTREMA"]
df_3["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO_DISCRETA"] = pd.cut(df_3["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"], bins=delta_bins, labels=delta_labels)

socioeconomic_labels = ["Q1_BAIXO", "Q2_MEDIO_BAIXO", "Q3_MEDIO_ALTO", "Q4_ALTO"]
df_3["IDH_MUNICIPAL_LONGEVIDADE_DISCRETA"] = pd.qcut(df_3["IDH_MUNICIPAL_LONGEVIDADE"], q=4, labels=socioeconomic_labels)
df_3["EXPECTATIVA_VIDA_DISCRETA"] = pd.qcut(df_3["EXPECTATIVA_VIDA"], q=4, labels=socioeconomic_labels)
df_3["PROPORCAO_POBRES_DISCRETA"] = pd.qcut(df_3["PROPORCAO_POBRES"], q=4, labels=socioeconomic_labels)

df_3 = df_3.drop(columns=["IDADE", "DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDH_MUNICIPAL_LONGEVIDADE", 
                          "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"])

In [ ]:
run_dt_pipeline(df_3, "Base 3 - Discretização")

# Os resultados obtidos na Base 3 evidenciam o impacto teórico da perda de granularidade dos dados no algoritmo de Árvore de Decisão. Ao discretizar variáveis contínuas cruciais (como IDADE e as métricas socioeconômicas) em intervalos fixos e quantis, limitou-se a capacidade do classificador de encontrar limiares ótimos e específicos de corte, resultando em uma redução sutil do F1-Score na validação cruzada (de 0.162 para 0.156) e no aumento de Falsos Positivos.

# Contudo, a Base 3 cumpre um papel metodológico fundamental: ela simplifica o espaço de estados do modelo, gerando uma estrutura 100% categórica que prioriza a interpretabilidade clínica. Em termos práticos de saúde pública, embora o desempenho estatístico bruto seja ligeiramente inferior ao da Base 1, as regras geradas na Base 3 tornam-se semanticamente inteligíveis para auditores humanos e gestores hospitalares, demonstrando o trade-off clássico entre a precisão matemática fina e a explicabilidade do modelo.

## Análise de Seleção de Variáveis 

### Feature Selection (Variáveis Preditoras x Variáveis Preditoras)

In [ ]:
df_1 = pd.read_csv("dados/BASE_v2.csv")
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
X = df_1.drop(columns=["ANO", "HOSPITALIZACAO"])

corr_matrix = X[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de Correlação X com X (Variáveis Numéricas)")
plt.show()

df_1 = df_1.drop(columns=["EXPECTATIVA_VIDA"])
# removeremos a variavel numerica EXPECTATIVA_VIDA

# Em resumo: Por que gastar energia com as numéricas e não com as categóricas? Nas categóricas: A redundância é parcial, o volume de cruzamentos é colossal (1.521 pares) e o nosso método embutido (a Árvore) já faz o trabalho sujo de ignorar as redundantes de forma 100% automatizada e otimizada matematicamente através da métrica de Gini.

### Feature Selection (Variáveis Preditoras x Variável Alvo)

Usou-se Chi-Quadrado, ANOVA F-Value e Feature Importance da Decision Tree

In [ ]:
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "PROPORCAO_POBRES"]
df_1 = pd.get_dummies(
    df_1, 
    columns=[col for col in df_1.columns if col not in numeric_cols + ["ANO", "HOSPITALIZACAO"]], 
    drop_first=True
)
categorical_cols = [col for col in df_1.columns if col not in numeric_cols + ["ANO", "HOSPITALIZACAO"]]

X = df_1.drop(columns=["ANO", "HOSPITALIZACAO"])
y = df_1["HOSPITALIZACAO"]

# Chi_Quadrado (Categóricas)
df_categorical = pd.DataFrame(index=categorical_cols)
chi2_scores, _ = chi2(X[categorical_cols], y)
df_categorical["CHI2_SCORE"] = chi2_scores
df_categorical["RANKING_ESTATISTICO"] = df_categorical["CHI2_SCORE"].rank(ascending=False, method="min").astype(int)

# ANOVA F-Value (Numéricas)
df_numeric = pd.DataFrame(index=numeric_cols)
anova_scores, _ = f_classif(X[numeric_cols], y)
df_numeric["ANOVA_SCORE"] = anova_scores
df_numeric["RANKING_ESTATISTICO"] = df_numeric["ANOVA_SCORE"].rank(ascending=False, method="min").astype(int)

# Concatenação dos Rankings
df_ranking = pd.concat([df_categorical[["RANKING_ESTATISTICO"]], df_numeric[["RANKING_ESTATISTICO"]]])

# Feature Importance da Decision Tree (Categóricas e Numéricas)
best_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=8,
    min_samples_leaf=20,
    min_samples_split=50,
    splitter="best",
    random_state=42
)
best_tree.fit(X, y)

df_tree = pd.DataFrame(index=X.columns)
df_tree["TREE_IMPORTANCE"] = best_tree.feature_importances_
df_tree["RANKING_TREE_IMPORTANCE"] = df_tree["TREE_IMPORTANCE"].rank(ascending=False, method="min").astype(int)

df_ranking = df_ranking.join(df_tree)
df_ranking["RANKING"] = df_ranking[["RANKING_ESTATISTICO", "RANKING_TREE_IMPORTANCE"]].mean(axis=1).rank(method="min").astype(int)
df_ranking = df_ranking.sort_values(by="RANKING")

df_ranking[["RANKING", "RANKING_ESTATISTICO", "RANKING_TREE_IMPORTANCE", "TREE_IMPORTANCE"]]

In [ ]:
num_cols= 15
top_features = df_ranking.head(num_cols).index.tolist()
df_4 = df_1[top_features + ["ANO", "HOSPITALIZACAO"]]

run_dt_pipeline(df_4, f"Base 4 - Feature Selection ({num_cols} Variáveis)", encode=False)
# Optou-se pela utilização da Base 4 (15 variáveis) em detrimento da Base 1 (todas as variáveis) porque ambos os cenários apresentaram um desempenho preditivo praticamente idêntico no teste (mesmo F1-Score de 0,17 e Recall de 61%).
# Pautando-se pelo princípio da parcimônia, a Base 4 é significativamente superior por dois motivos práticos: melhoria no desempenho computacional (ao eliminar dezenas de colunas que geravam ruído e redundância matemática) e viabilidade clínica. Em um cenário real de pronto-socorro, o modelo reduz drasticamente a quantidade de campos necessários para o médico preencher na triagem, tornando o sistema muito mais ágil, rápido e aplicável à rotina hospitalar sem perda de qualidade nos diagnósticos.

df_4.to_csv("dados/BASE_v3.csv", index=False)